# Exercise 16 (optional): A toy instability and its Landau damping

**Block 5: Landau damping, conceptually**

**Learning objectives**
* Build a crude "wakefield": a kick proportional to the beam centroid of the *previous* turn.
* See it drive a coherent oscillation that grows exponentially, and predict the growth rate.
* Add a tune spread with an octupole and find the strength above which the beam stays stable.

**Motivation.** Exercise 15 showed that a tune spread makes a coherent oscillation fade away. An instability
works the other way round: the beam's own fields keep pushing the centroid, coherently, turn after turn.
Which effect wins? This is the question that sets how strong the Landau octupoles in the LHC must be.

> **Honest warning.** This is a *toy model*: one bunch, one plane, a wake that remembers only one turn, and a
> ring made of one rotation plus one octupole kick. It reproduces the basic mechanism (growth without spread,
> stabilisation above a threshold spread of the order of the growth rate) but none of the details of real
> instabilities (frequency dependence of the impedance, head–tail modes, chromaticity...). The notes at the end
> list what was tried while preparing it.

> **Student version.** Fill in the cells marked `# YOUR CODE HERE` and run the check cells after them: they tell you when you are right.

In [ ]:
# --- setup: make `beamlib` importable --------------------------------------
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "beamlib").is_dir())
sys.path.insert(0, str(_root))

import numpy as np
import matplotlib.pyplot as plt
from beamlib import lattices as L, beams as B, plotting as P
from beamlib.plotting import COLORS, MM
P.setup()

BETA, EPS, Q0 = 50.0, 1e-6, 0.28        # same ring and beam as in Exercise 15
N = 5000                                # macro-particles
SIG = np.sqrt(BETA * EPS)

## 1. The toy wake

After every turn, each particle gets a kick proportional to where the beam centre was **one turn earlier**:

$$\Delta \hat p = -G\,\langle \hat x\rangle_{n-1}\qquad\Longleftrightarrow\qquad \Delta p_x = -\frac{G}{\beta}\,\langle x\rangle_{n-1}$$

$G$ is the (dimensionless) wake strength. Because of the one-turn delay, the kick is not in phase with the
current position: part of it acts like a push in the direction of motion, which feeds energy into the
coherent oscillation. For a beam without tune spread the centroid amplitude grows as $e^{r n}$ with

$$r = \frac{G}{2}\,\sin(2\pi Q) \quad \text{per turn}.$$

**Task.** Complete the `WakeKick` element: its `track` method applies the kick using the centroid stored
from the previous call, then stores the current centroid for the next turn. (Use `np.nanmean` so that lost
particles, set to NaN, are ignored.)

In [ ]:
class WakeKick:
    """Kick proportional to the beam centroid one turn earlier (a toy wakefield)."""

    def __init__(self, G: float, beta: float):
        self.G, self.beta = G, beta
        self.previous_xbar = 0.0      # centroid seen on the previous turn [m]
        self.history = []             # centroids, for plotting

    def track(self, p):
        p = p.copy()
        # YOUR CODE HERE
        raise NotImplementedError("Replace this line with your code")
        return p

In [ ]:
w = WakeKick(0.01, BETA)
p = np.array([[1e-3, 3e-3], [0, 0], [0, 0], [0, 0]])     # two particles, centroid at x = 2 mm
p1 = w.track(p)
assert np.allclose(p1, p) and np.isclose(w.previous_xbar, 2e-3), "first turn: no kick yet, centroid stored"
p2 = w.track(p)
assert np.allclose(p2[1, :], -0.01 / BETA * 2e-3), "second turn: kick from the stored centroid"
print("GOOD! WakeKick works")

## 2. A simulation function

`run(k3l, G, n_turns)` (given) tracks a matched Gaussian beam, initially displaced by 0.05 $\sigma$, through
"ring (rotation + octupole) $\to$ wake kick" and returns the centroid history. Particles with |$x$| > 10 cm are
considered lost. The simulation stops early if the centroid exceeds 3 $\sigma$.

**Task.** Complete `growth_rate(xbar)`: fit a straight line to log|centroid amplitude| against turn number.
To get the *amplitude* of an oscillating signal, take the maximum of |$\bar{x}$| over a sliding window of 8 turns
(given). Fit from turn 100 (the start is messy) to the end.

In [ ]:
def run(k3l: float, G: float, n_turns: int = 3000, seed: int = 1):
    rng = np.random.default_rng(seed)
    p = B.matched_gaussian(N, EPS, BETA, 0.0, rng=rng)
    p[:, 0] += 0.05 * SIG
    ring = L.HenonRing(Q0, 0.31, BETA, 0.0, 10.0, 0.0, knl=[0, 0, 0, k3l])
    wake = WakeKick(G, BETA)
    for _ in range(n_turns):
        p = wake.track(ring.one_turn_map(p))
        p[~(np.abs(p[:, 0]) < 0.1)] = np.nan
        if abs(wake.previous_xbar) > 3 * SIG:
            break
    return np.array(wake.history) / SIG, int(np.isnan(p[:, 0]).sum())


def envelope(xbar, window: int = 8):
    """Running maximum of |xbar| over `window` turns (amplitude of an oscillating signal)."""
    a = np.abs(xbar)
    return np.array([a[max(0, i - window + 1):i + 1].max() for i in range(len(a))])


def growth_rate(xbar, start: int = 100) -> float:
    """Exponential growth rate per turn of the centroid amplitude (negative: decay)."""
    env = envelope(xbar)
    # YOUR CODE HERE
    raise NotImplementedError("Replace this line with your code")

In [ ]:
t = np.arange(400)
assert abs(growth_rate(np.exp(0.01 * t) * np.cos(2 * np.pi * 0.28 * t)) - 0.01) < 1e-3
print("GOOD! growth_rate works")

## 3. Growth without tune spread

**Predict first.** With $G$ = 0.01 and $Q$ = 0.28, what growth rate do you expect from the formula? How many
turns does it take to grow by a factor $e$?

*Your prediction:*

In [ ]:
G = 0.01
r_theory = G / 2 * np.sin(2 * np.pi * Q0)
xbar0, lost0 = run(0.0, G)
r0 = growth_rate(xbar0)
print(f"growth rate: simulation {r0:.5f} /turn, formula {r_theory:.5f} /turn  (e-folding time {1 / r_theory:.0f} turns)")
assert abs(r0 / r_theory - 1) < 0.05

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(xbar0, color=COLORS["y"], lw=0.7, label="centroid, no octupole")
ax.plot(0.05 * np.exp(r_theory * np.arange(len(xbar0))), "k--", lw=1, label=r"$0.05 \exp(r n)$")
ax.set_xlabel("turn"); ax.set_ylabel(r"$\langle x\rangle / \sigma_x$"); ax.legend()
ax.set_title("toy wake, no tune spread: exponential growth")
plt.show()

## 4. Adding octupoles

**Predict first.** The octupole gives a tune spread of about $a \varepsilon = k_3L \beta^2 \varepsilon / 16\pi$ (Exercise 14).
A common rule of thumb says the beam becomes stable when the tune spread is larger than the growth rate
expressed in tune units, $r/2\pi$. Estimate the threshold $k_3L$.

*Your prediction:*

In [ ]:
# (about 15 s)
k3_values = np.array([0, 4, 8, 12, 16, 20, 25, 30, 40])
rates, losses, histories = [], [], {}
for k3 in k3_values:
    xb, nl = run(k3, G)
    rates.append(growth_rate(xb)); losses.append(nl); histories[k3] = xb
    print(f"k3L = {k3:4.0f} 1/m^3: tune spread a*eps = {k3 * BETA**2 * EPS / (16 * np.pi):.2e}, "
          f"growth rate = {rates[-1]:+.5f} /turn, {len(xb)} turns, lost {nl}")
rates = np.array(rates)

k3_threshold_estimate = (r_theory / (2 * np.pi)) * 16 * np.pi / (BETA**2 * EPS)
print(f"\nrule-of-thumb threshold: k3L ~ {k3_threshold_estimate:.1f} 1/m^3")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 4.5))
for k3, col in zip([0, 8, 16, 30], [COLORS["y"], COLORS["accent"], COLORS["x"], COLORS["sum"]]):
    axs[0].semilogy(envelope(histories[k3]), color=col, lw=1, label=rf"$k_3L$ = {k3}")
axs[0].set_xlabel("turn"); axs[0].set_ylabel(r"centroid amplitude / \sigma$"); axs[0].legend()
axs[0].set_title("centroid amplitude")
axs[1].plot(k3_values, rates * 1e3, "o-", color=COLORS["x"])
axs[1].axhline(0, color="k", lw=1)
axs[1].axvline(k3_threshold_estimate, color=COLORS["accent"], ls="--", label=r"rule of thumb: $a \varepsilon = r / 2\pi$")
axs[1].set_xlabel(r"octupole strength $k_3L$ [1/$\text{m}^3$]"); axs[1].set_ylabel(r"growth rate [$10^{-3}$ / turn]")
axs[1].legend(); axs[1].set_title("growth rate vs octupole strength")
plt.show()

assert rates[0] > 0.004, "unstable without octupole"
assert np.all(np.diff(rates[:7]) < 0), "growth slows down as the spread increases"
assert np.all(rates[k3_values >= 30] < 1e-4), "no growth well above threshold"
assert envelope(histories[30]).max() < 0.2, "the centroid stays near the noise level"

Without the octupole the coherent oscillation grows exponentially. As the octupole strength increases the
growth slows down. For intermediate strengths (around $k_3L \approx 16 \text{m}^{-3}$) the oscillation grows and then
*saturates* at a fraction of $\sigma$: a large coherent amplitude brings its own extra detuning. Above
$k_3L \approx 25–30 \text{ m}^{-3}$ the centroid stays at the noise level (a few % of $\sigma$, from the finite number of
macro-particles): the beam is **Landau damped**. The rule of thumb ($\approx 16 \text{ m}^{-3}$) gives the right order of
magnitude but is low by a factor $\approx$ 1.7 here, which is typical for such a crude criterion.

What happens physically: with a tune spread, only the particles whose tune is close to the coherent frequency
can follow the wake's push. The others oscillate at their own tunes and their contribution to the centroid
decoheres (Exercise 15) faster than the wake can make it grow.